# Superconductivity: FD discovery with mugen

This notebook runs `mugen` on the UCI superconductivity table. It explains each step, reports FD counts and timings, and compares the results with Desbordante on the same column prefixes.

The table is `superconductivty+data.zip` extracted to `superconductivity/train.csv`. It has 21,263 superconducting materials, 81 numeric features, and the target `critical_temp`.

Sections: 1 setup, 2 dataset, 3 why the search gets expensive, 4 LHS cap on all columns, 5 scaling with column count, 6 Desbordante on the same prefixes, 7 parity check, 8 summary table, 9 what the FDs say about the target, 10 optional full run, 11 caveats and conclusion.

All timings come from one machine, so absolute times will differ elsewhere. The ratios are more stable.

## 1. Setup

We import `mugen`, the Desbordante package, and Polars. We print their versions and the path of the `mugen` module. The module path confirms the notebook uses the `.venv` build, not an older copy.

In [1]:
import importlib.metadata as md
import json
import subprocess
import sys
import time
from pathlib import Path

import desbordante
import mugen
import pandas as pd
import polars as pl

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'superconductivity' / 'train.csv').exists())
DATA = REPO / 'superconductivity' / 'train.csv'

print('desbordante', md.version('desbordante'))
print('polars', md.version('polars'), '| pandas', md.version('pandas'))
print('mugen loaded from', mugen.__file__)
print('data file', DATA)

desbordante 2.5.0
polars 2.0.0 | pandas 2.3.3
mugen loaded from /home/alonso/mugen/.venv/lib/python3.10/site-packages/mugen/__init__.py
data file /home/alonso/mugen/superconductivity/train.csv


## 2. The dataset

Each row is one superconducting material. The features are statistics computed from the chemical elements in the material, such as mean atomic mass, valence, and thermal conductivity. The target `critical_temp` is the temperature below which the material superconducts.

This makes a useful FD test. The columns are numeric floats with many distinct values, and there are no nulls, so null handling does not affect the results here. Many features are derived from the same few elements, so they may be related to each other. Real FDs over floats are exact equalities, so a near-duplicate value does not count as a dependency.

We load the table with Polars. The target is the last column, and we keep it in the search so its FDs can be read later.

In [2]:
df = pl.read_csv(DATA)
target = 'critical_temp'
features = [c for c in df.columns if c != target]

print('rows, columns:', df.shape)
print('feature columns:', len(features), '| target:', target, '| target is last column:', df.columns[-1] == target)
print('null values in table:', sum(df.null_count().row(0)))
print(df.select(features[:4]).head(3))

rows, columns: (21263, 82)
feature columns: 81 | target: critical_temp | target is last column: True
null values in table: 0
shape: (3, 4)
┌────────────────────┬──────────────────┬──────────────────────┬───────────────────┐
│ number_of_elements ┆ mean_atomic_mass ┆ wtd_mean_atomic_mass ┆ gmean_atomic_mass │
│ ---                ┆ ---              ┆ ---                  ┆ ---               │
│ i64                ┆ f64              ┆ f64                  ┆ f64               │
╞════════════════════╪══════════════════╪══════════════════════╪═══════════════════╡
│ 4                  ┆ 88.944468        ┆ 57.862692            ┆ 66.361592         │
│ 5                  ┆ 92.729214        ┆ 58.518416            ┆ 73.132787         │
│ 4                  ┆ 88.944468        ┆ 57.885242            ┆ 66.361592         │
└────────────────────┴──────────────────┴──────────────────────┴───────────────────┘


In [3]:
distinct = sorted(df.select(pl.all().n_unique()).row(0))
print('distinct values per column: min', distinct[0], '| median', distinct[len(distinct) // 2], '| max', distinct[-1])

distinct values per column: min 7 | median 3365 | max 15165


## 3. Why FD discovery gets expensive

An FD `X -> A` says that any two rows with the same values in the columns `X` also have the same value in `A`. A minimal FD has no smaller left-hand side that still determines `A`.

To find every minimal FD, the search has to consider sets of columns as candidate left-hand sides. With `n` columns, each right-hand side has `n - 1` other columns, so there are `2^(n-1)` possible left-hand sides for it. The search prunes most of these, but the worst case still grows exponentially with the number of columns. The table below shows the size of that space for each width we use.

In [4]:
for n in [10, 20, 30, 40, 50, 60, 81]:
    print(f'{n:>3} columns: {2 ** (n - 1):>28,} possible left-hand sides for one right-hand side')

 10 columns:                          512 possible left-hand sides for one right-hand side
 20 columns:                      524,288 possible left-hand sides for one right-hand side
 30 columns:                  536,870,912 possible left-hand sides for one right-hand side
 40 columns:              549,755,813,888 possible left-hand sides for one right-hand side
 50 columns:          562,949,953,421,312 possible left-hand sides for one right-hand side
 60 columns:      576,460,752,303,423,488 possible left-hand sides for one right-hand side
 81 columns: 1,208,925,819,614,629,174,706,176 possible left-hand sides for one right-hand side


## 4. Capping the left-hand side across all columns

`max_lhs` limits how many columns a left-hand side can have. A minimal FD with at most `k` columns on the left depends only on candidates of size at most `k`. So a run with `max_lhs=k` returns exactly the uncapped minimal FDs whose left-hand side has at most `k` columns. The cap makes the search cheaper, and the result is a subset of the uncapped output, not an approximation.

This runs the search over all 82 columns, target included, at caps 1, 2, and 3. Caps 1 and 2 are repeated three times and the best time is kept. Cap 3 is a single run, because it takes longer. `threads` is left unset, so mugen uses all cores.

In [5]:
def best_of(fn, rounds=3):
    best, result = float('inf'), None
    for _ in range(rounds):
        start = time.perf_counter()
        result = fn()
        best = min(best, time.perf_counter() - start)
    return best, result


RESULTS = {}
for cap, rounds in [(1, 3), (2, 3), (3, 1)]:
    t, (names, fds) = best_of(lambda: mugen.discover_fds(df, max_lhs=cap), rounds)
    RESULTS[cap] = (names, fds)
    print(f'max_lhs={cap}: {len(fds):>9,} minimal FDs in {t:8.3f} s (best of {rounds})')

max_lhs=1:       289 minimal FDs in    0.880 s (best of 3)


max_lhs=2:    34,824 minimal FDs in    8.225 s (best of 3)


max_lhs=3:   281,332 minimal FDs in   64.024 s (best of 1)


## 5. Scaling with the number of columns

This section runs mugen without a cap on the first `k` feature columns, for `k` from 10 to 60. Each run uses all threads. For `k` up to 40 it also runs on one thread, which shows how much the parallel search helps.

Each call is best of three. Discovery covers the Arrow handoff and the search. The one-thread runs stop at 40 columns, because longer single-thread runs would take minutes each. The set of FDs is stored as a set of column names, so it can be compared with Desbordante in section 7.

In [6]:
def fd_set(names, fds):
    return {(frozenset(names[i] for i in lhs), names[rhs]) for lhs, rhs in fds}


MUGEN = {}
for k in [10, 20, 30, 40, 50, 60]:
    sub = df.select(features[:k])
    t_all, (names, fds) = best_of(lambda: mugen.discover_fds(sub), 3)
    t_one = best_of(lambda: mugen.discover_fds(sub, threads=1), 3)[0] if k <= 40 else None
    MUGEN[k] = {'all': t_all, 'one': t_one, 'fds': len(fds), 'set': fd_set(names, fds)}
    one = f'{t_one:8.3f}' if t_one is not None else '     n/a'
    print(f'{k:>3} columns: {len(fds):>9,} FDs | 1 thread {one} s | all threads {t_all:8.3f} s')

 10 columns:        59 FDs | 1 thread    0.043 s | all threads    0.022 s


 20 columns:       585 FDs | 1 thread    0.371 s | all threads    0.084 s


 30 columns:     3,584 FDs | 1 thread    2.320 s | all threads    0.327 s


 40 columns:    12,886 FDs | 1 thread    7.279 s | all threads    0.857 s


 50 columns:    45,752 FDs | 1 thread      n/a s | all threads    2.074 s


 60 columns:   123,159 FDs | 1 thread      n/a s | all threads    6.912 s


## 6. Desbordante on the same column prefixes

Desbordante runs in a separate Python process with a time limit. A slow Desbordante run then cannot hang the kernel, and a timeout is reported as a timeout, not as a time. Each Desbordante run uses one thread, which matches the single-thread mugen runs.

The Desbordante timer covers `load_data` and `execute`, which is the same span as the mugen timer. The CSV read is outside the timer. Each Desbordante run is a single run, so it is noisier than the best-of-three mugen times. The time limit is 120 seconds per run.

The last line also runs mugen on the 81 features alone with `max_lhs=1`, so it can be compared with Desbordante on the same input. The earlier caps in section 4 include the target.

In [7]:
DESB_WORKER = '''
import json, sys, time
import pandas as pd
import desbordante

path, k, cap = sys.argv[1], int(sys.argv[2]), sys.argv[3]
frame = pd.read_csv(path).iloc[:, :k]
start = time.perf_counter()
algo = desbordante.fd.algorithms.Pyro()
algo.load_data(table=frame)
kwargs = {'error': 0.0, 'seed': 0, 'threads': 1}
if cap != 'none':
    kwargs['max_lhs'] = int(cap)
algo.execute(**kwargs)
elapsed = time.perf_counter() - start
fds = [[list(lhs), rhs] for lhs, rhs in (fd.to_name_tuple() for fd in algo.get_fds())]
print(json.dumps({'time': elapsed, 'fds': fds}))
'''


def run_desbordante(k, timeout, cap='none'):
    cmd = [sys.executable, '-c', DESB_WORKER, str(DATA), str(k), cap]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return {'status': 'timeout', 'timeout_s': timeout}
    if proc.returncode != 0:
        return {'status': 'error', 'stderr': proc.stderr[-1000:]}
    return {'status': 'ok', **json.loads(proc.stdout.strip().splitlines()[-1])}

In [8]:
DESB = {}
for k in [10, 20, 30, 40, 50]:
    DESB[k] = run_desbordante(k, timeout=120)
    if DESB[k]['status'] == 'ok':
        print(f"{k:>3} columns: {len(DESB[k]['fds']):>9,} FDs in {DESB[k]['time']:8.3f} s (1 run)")
    else:
        print(f"{k:>3} columns: {DESB[k]['status']} after {DESB[k]['timeout_s']} s")

cap1_desb = run_desbordante(81, timeout=120, cap='1')
if cap1_desb['status'] == 'ok':
    print(f"Desbordante, 81 features, max_lhs=1: {len(cap1_desb['fds'])} FDs in {cap1_desb['time']:.3f} s")
else:
    print(f"Desbordante, 81 features, max_lhs=1: {cap1_desb['status']} after {cap1_desb['timeout_s']} s")

t_cap1, (n81, f81) = best_of(lambda: mugen.discover_fds(df.select(features), max_lhs=1), 3)
print(f'mugen, 81 features, max_lhs=1: {len(f81)} FDs in {t_cap1:.3f} s (best of 3)')

 10 columns:        59 FDs in    0.172 s (1 run)


 20 columns:       585 FDs in    0.584 s (1 run)


 30 columns:     3,584 FDs in    2.360 s (1 run)


 40 columns:    12,886 FDs in   18.193 s (1 run)


 50 columns: timeout after 120 s


Desbordante, 81 features, max_lhs=1: timeout after 120 s


mugen, 81 features, max_lhs=1: 289 FDs in 0.831 s (best of 3)


## 7. Are the FDs the same?

Speed only matters if the answers match. We compare the FD sets by column name, not by index, so the comparison does not depend on the column order in either library. A dependency is a pair of a left-hand side set and a right-hand side column.

The check covers the uncapped prefixes from 10 to 40 columns, where Desbordante finished.

In [9]:
for k in [10, 20, 30, 40]:
    d = DESB[k]
    if d['status'] != 'ok':
        print(f"{k:>3} columns: Desbordante {d['status']}, not compared")
        continue
    desb_set = {(frozenset(lhs), rhs) for lhs, rhs in d['fds']}
    same = desb_set == MUGEN[k]['set']
    print(f"{k:>3} columns: mugen {len(MUGEN[k]['set']):>7,} | desbordante {len(desb_set):>7,} | identical: {same}")

 10 columns: mugen      59 | desbordante      59 | identical: True
 20 columns: mugen     585 | desbordante     585 | identical: True
 30 columns: mugen   3,584 | desbordante   3,584 | identical: True
 40 columns: mugen  12,886 | desbordante  12,886 | identical: True


## 8. Summary of timings

This table gathers the runs above. The speedup compares Desbordante's single run with mugen's best run on all threads. It does not compare the one-thread mugen times. `n/a` means the run was not measured, and a status such as `timeout` means Desbordante did not finish within the limit.

In [10]:
rows = []
for k in [10, 20, 30, 40, 50, 60]:
    m = MUGEN[k]
    d = DESB.get(k, {'status': 'not run'})
    if d['status'] == 'ok':
        desb_time = round(d['time'], 3)
        speedup = round(d['time'] / m['all'], 1)
    else:
        desb_time = d['status']
        speedup = 'n/a'
    rows.append({
        'columns': k,
        'FDs': m['fds'],
        'mugen 1 thread (s)': round(m['one'], 3) if m['one'] is not None else 'n/a',
        'mugen all threads (s)': round(m['all'], 3),
        'Desbordante 1 thread (s)': desb_time,
        'speedup (all threads)': speedup,
    })
print(pd.DataFrame(rows).to_string(index=False))

 columns    FDs mugen 1 thread (s)  mugen all threads (s) Desbordante 1 thread (s) speedup (all threads)
      10     59              0.043                  0.022                    0.172                   7.7
      20    585              0.371                  0.084                    0.584                   6.9
      30   3584               2.32                  0.327                     2.36                   7.2
      40  12886              7.279                  0.857                   18.193                  21.2
      50  45752                n/a                  2.074                  timeout                   n/a
      60 123159                n/a                  6.912                  not run                   n/a


## 9. Reading the results: FDs that determine the critical temperature

This is the question the table is built for: which features fix the critical temperature? The capped run at `max_lhs=2` from section 4 includes every minimal FD with up to two left-hand columns. The count below is the number of those with `critical_temp` on the right. The output is zero, so no single feature or pair of features determines the critical temperature exactly in this table. The cap-3 run and the uncapped run could find FDs with larger left-hand sides, but this notebook does not list them.

An exact FD would mean the values agree exactly in this table. It would not show a physical law, and it would not guarantee the relationship holds for new materials. Because the result is zero, the useful conclusion is that the critical temperature needs several features together, or a statistical model, rather than a simple exact rule.

In [11]:
names2, fds2 = RESULTS[2]
tidx = names2.index(target)
to_target = [(lhs, rhs) for lhs, rhs in fds2 if rhs == tidx]
one_col = [lhs for lhs, _ in to_target if len(lhs) == 1]
print(f'minimal FDs with {target} on the right, left-hand side up to 2 columns: {len(to_target)}')
print(f'of which the left-hand side has a single column: {len(one_col)}')
for lhs, rhs in sorted(to_target, key=lambda f: (len(f[0]), f[0]))[:10]:
    print('  {' + ', '.join(names2[i] for i in lhs) + '} -> ' + names2[rhs])

minimal FDs with critical_temp on the right, left-hand side up to 2 columns: 0
of which the left-hand side has a single column: 0


## 10. Optional: full uncapped run on all 81 features

Without a cap the search has to find every minimal FD, which is the costliest run. It is off by default, because a single run takes several minutes. Set `RUN_FULL = True` to run it.

The result from an earlier run outside this notebook, with all threads, was 534 seconds and 1,038,258 minimal FDs. The result is also large: about a million FD tuples are built in Python, and this notebook does not measure how much of the time that takes.

In [12]:
RUN_FULL = False

if RUN_FULL:
    start = time.perf_counter()
    names_full, fds_full = mugen.discover_fds(df.select(features))
    elapsed = time.perf_counter() - start
    print(f'{len(features)} features, uncapped: {len(fds_full):,} minimal FDs in {elapsed:.1f} s')
else:
    print('Skipped. Set RUN_FULL = True to run it. An earlier run took 534 s with all threads and found 1,038,258 minimal FDs.')

Skipped. Set RUN_FULL = True to run it. An earlier run took 534 s with all threads and found 1,038,258 minimal FDs.


## 11. Caveats and conclusion

**Caveats**

- Floats are compared exactly. Two values that differ in the last digit are different values, so a near-dependency is not an FD.
- Desbordante runs once per size, with a 120-second limit. Its 50-column timeout means it did not finish within that limit. A longer limit might let it finish, and this notebook does not show that.
- mugen is not measured on one thread above 40 columns, so the one-thread column is blank there.
- The cap-3 run and the 81-feature cap-1 comparison are single runs or single Desbordante runs, so they carry more noise.
- Memory is not measured in this notebook.
- All timings come from one machine. Thread counts and the timing span matter, so compare the ratios more than the absolute numbers.

**Conclusion**

- mugen returns the same named FD sets as Desbordante on the four uncapped prefixes (10 to 40 columns) where Desbordante finished.
- Against one-thread Desbordante, mugen on all threads is about 7 times faster at 10 to 30 columns and about 21 times faster at 40 columns. The 40-column ratio is the largest measured, and the ratios come from single Desbordante runs.
- The LHS cap is the practical tool for the wide table. Caps 1 and 2 finish in seconds, and cap 3 takes about a minute (64 s in this run) on all 82 columns. Desbordante did not finish at cap 1 on 81 features within the limit.
- The uncapped search over all 81 features is possible, but it produces over a million minimal FDs and takes several minutes. This notebook does not separate the search cost from the cost of building the output, so both are part of the total.
- The exact FD search finds no FD with `critical_temp` on the right at up to two left-hand columns.